# Lesson 9 — Production Best Practices

## What you will learn
9 practices that separate **"works on my machine"** from **"works in production"**:

| # | Practice | Why it matters |
|---|----------|----------------|
| 1 | **Logging** | Trace every node execution |
| 2 | **Input validation** | Reject bad inputs early |
| 3 | **Structured output** | Force LLM to return valid JSON/objects |
| 4 | **Retry + error handling** | Handle LLM failures gracefully |
| 5 | **Streaming** | Better UX — show tokens as they arrive |
| 6 | **Recursion limit** | Prevent runaway loops |
| 7 | **Parallel execution** | Speed up with `Send()` |
| 8 | **Full production graph** | All practices combined |
| 9 | **Checklist** | Reference for every production project |

In [ ]:
import logging
import time
from typing import Annotated, Optional
from typing_extensions import TypedDict
from pydantic import BaseModel, ValidationError
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.types import Send

llm = ChatOllama(model="llama3", temperature=0)

---
## Practice 1 — Structured Logging

Call `logger.info()` at the **start of every node** so you can trace what happened.

In [ ]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger("langgraph")

def log_node(name: str, state: dict):
    logger.info(f"NODE={name} | msgs={len(state.get('messages', []))} | keys={list(state.keys())}")

print("Logger configured!")

---
## Practice 2 — Input Validation with Pydantic

In [ ]:
class UserQuery(BaseModel):
    question:   str
    max_tokens: Optional[int] = 512
    language:   Optional[str] = "english"

    def model_post_init(self, __context):
        if len(self.question.strip()) < 3:
            raise ValueError("Question too short (min 3 chars)")
        if len(self.question) > 2000:
            raise ValueError("Question too long (max 2000 chars)")


# Valid input
q = UserQuery(question="What is LangGraph?", language="english")
print(f"Valid: {q}")

# Invalid input
try:
    bad = UserQuery(question="Hi")  # too short
except ValidationError as e:
    print(f"Rejected: {e.errors()[0]['msg']}")

---
## Practice 3 — Structured Output

`with_structured_output()` forces the LLM to return a valid Pydantic object.  
**Never parse LLM JSON by hand** — use this instead.

In [ ]:
class SentimentResult(BaseModel):
    sentiment:   str          # positive / negative / neutral
    confidence:  float        # 0.0 to 1.0
    key_phrases: list[str]    # top phrases that led to this result
    summary:     str          # one sentence


structured_llm = llm.with_structured_output(SentimentResult)

result = structured_llm.invoke([
    SystemMessage(content="Analyze sentiment of the text. Return structured data."),
    HumanMessage(content="This product is absolutely amazing! Best purchase I've ever made!")
])

# result is a GUARANTEED valid SentimentResult object
print(f"Sentiment:   {result.sentiment}")
print(f"Confidence:  {result.confidence:.2f}")
print(f"Key phrases: {result.key_phrases}")
print(f"Summary:     {result.summary}")

---
## Practice 4 — Retry + Error Handling in Nodes

Never let a single LLM failure crash your graph.  
Use exponential backoff and a graceful fallback.

In [ ]:
class ResilientState(TypedDict):
    messages:    Annotated[list, add_messages]
    retry_count: int
    error:       str


def resilient_node(state: ResilientState) -> dict:
    log_node("resilient_node", state)
    MAX_RETRIES = 3

    for attempt in range(MAX_RETRIES):
        try:
            logger.info(f"  Attempt {attempt+1}/{MAX_RETRIES}")
            response = llm.invoke(state["messages"])
            logger.info("  LLM call succeeded")
            return {"messages": [response], "error": ""}
        except Exception as e:
            logger.warning(f"  Attempt {attempt+1} failed: {e}")
            if attempt < MAX_RETRIES - 1:
                wait = 2 ** attempt   # 1s → 2s → 4s
                logger.info(f"  Retrying in {wait}s")
                time.sleep(wait)
            else:
                return {"error": str(e), "messages": [AIMessage(content="I encountered an error. Please try again.")]}


def error_handler(state: ResilientState) -> dict:
    log_node("error_handler", state)
    logger.error(f"Handling error: {state['error']}")
    return {"messages": [AIMessage(content=f"Error handled: {state['error']}")], "error": ""}


b = StateGraph(ResilientState)
b.add_node("agent",  resilient_node)
b.add_node("errors", error_handler)
b.add_edge(START, "agent")
b.add_conditional_edges("agent",
    lambda s: "errors" if s.get("error") else "end",
    {"errors": "errors", "end": END})
b.add_edge("errors", END)
resilient_graph = b.compile()

result = resilient_graph.invoke({
    "messages": [HumanMessage(content="What are the benefits of retry logic in production AI?")],
    "retry_count": 0, "error": ""
})
print(result["messages"][-1].content[:200])

---
## Practice 5 — Streaming

Show tokens as they arrive. Critical for chatbot UX.

In [ ]:
class SimpleState(TypedDict):
    messages: Annotated[list, add_messages]

b_s = StateGraph(SimpleState)
b_s.add_node("chat", lambda s: {"messages": [llm.invoke(s["messages"])]})
b_s.add_edge(START, "chat")
b_s.add_edge("chat", END)
stream_graph = b_s.compile()

print("Streaming response:")
print("-" * 40)
for chunk, metadata in stream_graph.stream(
    {"messages": [HumanMessage(content="Write a haiku about Python programming.")]},
    stream_mode="messages"
):
    if chunk.content:
        print(chunk.content, end="", flush=True)
print("\n" + "-" * 40)

---
## Practice 6 — Recursion Limit

Always set `recursion_limit` in config to prevent runaway graphs.

In [ ]:
class CountState(TypedDict):
    messages: Annotated[list, add_messages]
    count: int

b_l = StateGraph(CountState)
b_l.add_node("loop", lambda s: {"count": s["count"] + 1})
b_l.add_edge(START, "loop")
b_l.add_conditional_edges("loop",
    lambda s: "end" if s["count"] >= 3 else "loop",
    {"loop": "loop", "end": END})
loop_graph = b_l.compile()

# recursion_limit = max steps before GraphRecursionError is raised
result = loop_graph.invoke({"messages": [], "count": 0}, config={"recursion_limit": 10})
print(f"Loop finished at count={result['count']}  (limit was 10)")

---
## Practice 7 — Parallel Execution with Send()

`Send()` fans out to multiple nodes in **parallel** — great for batch processing.

In [ ]:
class ParallelState(TypedDict):
    topics:    list[str]
    summaries: Annotated[list, lambda x, y: x + y]  # merge by appending


def summarize_one(state: dict) -> dict:
    topic = state["topic"]
    resp  = llm.invoke([HumanMessage(content=f"In one sentence, what is {topic}?")])
    return {"summaries": [f"{topic}: {resp.content}"]}


b_p = StateGraph(ParallelState)
b_p.add_node("summarize", summarize_one)
# Send() launches one "summarize" task per topic, all in parallel
b_p.add_conditional_edges(START, lambda s: [Send("summarize", {"topic": t}) for t in s["topics"]], ["summarize"])
b_p.add_edge("summarize", END)
par_graph = b_p.compile()

result = par_graph.invoke({
    "topics":    ["LangGraph", "SQLite", "Python decorators"],
    "summaries": []
})
print("Parallel results:")
for s in result["summaries"]:
    print(f"  • {s[:100]}")

---
## Practice 9 — Production Checklist

Copy this into every new LangGraph project:

```python
# ✅ PRODUCTION CHECKLIST
#
# BEFORE BUILDING:
#   ✅ Define state with TypedDict — all fields explicit
#   ✅ Validate inputs with Pydantic before invoking
#
# IN EVERY NODE:
#   ✅ log_node(name, state) at the start
#   ✅ try/except with retry + exponential backoff
#   ✅ Return only the keys you update (not the full state)
#
# GRAPH SETUP:
#   ✅ Add error_handler node as safety net
#   ✅ Set recursion_limit in config (default: 25)
#   ✅ Use checkpointer for any stateful/multi-user app
#   ✅ Use thread_id per user/session
#
# LLM CALLS:
#   ✅ temperature=0 for agents, higher for creative tasks
#   ✅ with_structured_output() when you need reliable JSON
#   ✅ Use streaming for chat interfaces
#
# TOOLS:
#   ✅ Docstrings are critical — LLM reads them
#   ✅ Read-only guard on database tools
#   ✅ Sensitive tools require HITL (interrupt)
#
# PERFORMANCE:
#   ✅ Use Send() for parallel workloads
#   ✅ Limit message history (trim to last N messages)
```